In [1]:
import sys
print("python exe:", sys.executable)   # full path to the interpreter
print("python ver:", sys.version)

# try torch only in the working notebook
try:
    import torch
    print("torch      :", torch.__version__, torch.__file__)
except ModuleNotFoundError as e:
    print("torch not importable:", e)



python exe: c:\Users\aneek\anaconda3\envs\tf_gpu_env\python.exe
python ver: 3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:49:16) [MSC v.1929 64 bit (AMD64)]
torch      : 1.12.1+cu113 c:\Users\aneek\anaconda3\envs\tf_gpu_env\lib\site-packages\torch\__init__.py


c:\Users\aneek\anaconda3\envs\tf_gpu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import tensorflow as tf
print(tf.__version__)  # This should print the version of TensorFlow
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

print("CUDA version:", tf.sysconfig.get_build_info()["cuda_version"])
print("cuDNN version:", tf.sysconfig.get_build_info()["cudnn_version"])

from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

2.10.0
Num GPUs Available:  1
CUDA version: 64_112
cuDNN version: 64_8
[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 10358477183655416882
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 5713690624
locality {
  bus_id: 1
  links {
  }
}
incarnation: 12963997954659321067
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9"
xla_global_id: 416903419
]


In [3]:
import time
import tensorflow as tf
import psutil

class PowerMonitor:
    def __init__(self):
        self.gpu_available = tf.config.list_physical_devices('GPU')
        
        # Hardware power specifications (adjust these values for your system)
        self.cpu_tdp = 65    # Typical TDP for desktop CPUs in watts
        self.gpu_tdp = 250   # Typical TDP for desktop GPUs in watts
        
    def get_stats(self):
        """Get system stats with power estimation"""
        stats = {
            'timestamp': time.time(),
            'cpu_%': psutil.cpu_percent(interval=0.1),
            'ram_mb': psutil.virtual_memory().used / (1024**2),
            'gpu_mem_mb': 0,
            'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85  # Base CPU power
        }
        
        if self.gpu_available:
            try:
                # TensorFlow GPU memory monitoring
                mem_info = tf.config.experimental.get_memory_info('GPU:0')
                stats.update({
                    'gpu_mem_mb': mem_info['current'] / (1024**2),
                    'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85 + 
                              self.gpu_tdp * 0.5 * 0.75  # Add GPU power estimate
                })
            except:
                pass
                
        return stats

# Initialize monitor
monitor = PowerMonitor()

In [4]:
import time
import torch
import psutil
import os

class PowerMonitor1:
    def __init__(self):
        self.gpu_available = torch.cuda.is_available()
        self.process = psutil.Process(os.getpid())  # Track current process
        
        # Hardware power specifications
        self.cpu_tdp = 65
        self.gpu_tdp = 250
        
    def get_stats(self):
        """Get process-specific stats with power estimation"""
        process_memory = self.process.memory_info()
        
        stats = {
            'timestamp': time.time(),
            'cpu_%': psutil.cpu_percent(interval=0.1),
            'process_ram_mb': process_memory.rss / (1024**2),  # Only this process's RAM
            'gpu_mem_mb': 0,
            'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85
        }
        
        if self.gpu_available:
            try:
                gpu_memory_allocated = torch.cuda.memory_allocated()
                stats.update({
                    'gpu_mem_mb': gpu_memory_allocated / (1024**2),
                    'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85 + 
                              self.gpu_tdp * 0.5 * 0.75
                })
            except Exception as e:
                print(f"Error retrieving GPU memory: {e}")
                
        return stats

# Initialize monitor1
monitor1 = PowerMonitor1()

# Model

In [5]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import timm

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from PIL import Image


# ============================================================
# CONFIGURATION
# ============================================================

INPUT_SIZE = 160
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 1e-4
RANDOM_SEED = 42

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# LOAD PRETRAINED CROSSVIT-15
# ============================================================

# Current timm identifier
MODEL_NAME = "crossvit_15_240.in1k"

try:
    model = timm.create_model(
        MODEL_NAME,
        pretrained=True,
        num_classes=2
    )

except RuntimeError:
    # Compatibility with older timm versions
    MODEL_NAME = "crossvit_15_240"

    model = timm.create_model(
        MODEL_NAME,
        pretrained=True,
        num_classes=2
    )

print("Loaded model:", MODEL_NAME)


# ============================================================
# ADAPT CROSSVIT POSITIONAL EMBEDDINGS TO 160 × 160
# ============================================================

def resize_position_embedding(
    position_embedding,
    old_grid_size,
    new_grid_size
):
    """
    Resize a CrossViT positional embedding using bicubic
    interpolation while retaining its class token.
    """

    class_token_embedding = position_embedding[:, :1, :]
    patch_embeddings = position_embedding[:, 1:, :]

    old_height, old_width = old_grid_size
    new_height, new_width = new_grid_size

    expected_old_patches = old_height * old_width

    if patch_embeddings.shape[1] != expected_old_patches:
        raise ValueError(
            "Position-embedding shape does not match the "
            f"expected old grid {old_grid_size}. "
            f"Found {patch_embeddings.shape[1]} patch tokens."
        )

    embedding_dimension = patch_embeddings.shape[-1]

    patch_embeddings = patch_embeddings.reshape(
        1,
        old_height,
        old_width,
        embedding_dimension
    )

    patch_embeddings = patch_embeddings.permute(
        0,
        3,
        1,
        2
    )

    patch_embeddings = F.interpolate(
        patch_embeddings,
        size=(new_height, new_width),
        mode="bicubic",
        align_corners=False
    )

    patch_embeddings = patch_embeddings.permute(
        0,
        2,
        3,
        1
    )

    patch_embeddings = patch_embeddings.reshape(
        1,
        new_height * new_width,
        embedding_dimension
    )

    return torch.cat(
        [
            class_token_embedding,
            patch_embeddings
        ],
        dim=1
    )


def adapt_crossvit_input_size(
    model,
    new_input_size=160
):
    """
    Adapt pretrained CrossViT-15 from its native 240-pixel
    input to a 160-pixel external input.

    CrossViT branch 1 retains the model's original internal
    image-scale ratio.
    """

    if not hasattr(model, "patch_embed"):
        raise AttributeError(
            "The supplied model does not appear to be CrossViT."
        )

    if len(model.patch_embed) != 2:
        raise ValueError(
            "This function expects a two-branch CrossViT model."
        )

    # Record original branch dimensions
    old_branch_sizes = [
        tuple(patch_embed.img_size)
        for patch_embed in model.patch_embed
    ]

    # Retain CrossViT's original scale ratio between branches
    second_branch_scale = (
        old_branch_sizes[1][0]
        / old_branch_sizes[0][0]
    )

    new_branch_sizes = [
        (new_input_size, new_input_size),

        (
            int(new_input_size * second_branch_scale),
            int(new_input_size * second_branch_scale)
        )
    ]

    for branch_index, patch_embed in enumerate(
        model.patch_embed
    ):
        patch_size = tuple(
            int(value)
            for value in patch_embed.patch_size
        )

        old_grid_size = (
            old_branch_sizes[branch_index][0]
            // patch_size[0],

            old_branch_sizes[branch_index][1]
            // patch_size[1]
        )

        new_grid_size = (
            new_branch_sizes[branch_index][0]
            // patch_size[0],

            new_branch_sizes[branch_index][1]
            // patch_size[1]
        )

        position_name = (
            f"pos_embed_{branch_index}"
        )

        old_position_embedding = getattr(
            model,
            position_name
        )

        resized_position_embedding = (
            resize_position_embedding(
                old_position_embedding.detach(),
                old_grid_size,
                new_grid_size
            )
        )

        setattr(
            model,
            position_name,
            nn.Parameter(
                resized_position_embedding
            )
        )

        # Update the expected input size for this branch
        patch_embed.img_size = (
            new_branch_sizes[branch_index]
        )

        patch_embed.num_patches = (
            new_grid_size[0]
            * new_grid_size[1]
        )

        print(
            f"Branch {branch_index}: "
            f"{old_branch_sizes[branch_index]} "
            f"-> {new_branch_sizes[branch_index]} | "
            f"grid {old_grid_size} -> {new_grid_size}"
        )

    model.img_size = (
        new_input_size,
        new_input_size
    )

    model.img_size_scaled = (
        new_branch_sizes
    )

    return model


model = adapt_crossvit_input_size(
    model,
    new_input_size=INPUT_SIZE
)

model = model.to(device)

print("\nCrossViT adapted successfully.")

# All pretrained parameters should remain trainable
for parameter in model.parameters():
    parameter.requires_grad = True


trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Trainable parameters:", trainable_parameters)
print("Total parameters:", total_parameters)


# Verify the adapted model before creating the full loaders
model.eval()

with torch.inference_mode():
    sample_input = torch.randn(
        2,
        3,
        INPUT_SIZE,
        INPUT_SIZE,
        device=device
    )

    sample_output = model(sample_input)

print("Test output shape:", sample_output.shape)
# ============================================================
# CROSSVIT TRANSFORM
# ============================================================

crossvit_transform = transforms.Compose([
    transforms.Resize(
        (INPUT_SIZE, INPUT_SIZE),
        interpolation=InterpolationMode.BICUBIC
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])
# ============================================================
# CROSSVIT DATASET
# ============================================================

class DeepfakeCrossViTDataset(Dataset):
    """
    Dataset for OpenCV-loaded deepfake images.

    Images:
        BGR uint8 arrays with shape (160, 160, 3)

    Labels:
        0 = real
        1 = fake
    """

    def __init__(
        self,
        images,
        labels,
        transform
    ):
        self.images = images

        self.labels = np.asarray(
            labels,
            dtype=np.int64
        )

        self.transform = transform

        if len(self.images) != len(self.labels):
            raise ValueError(
                "The numbers of images and labels do not match."
            )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image = self.images[index]
        label = int(self.labels[index])

        if isinstance(image, np.ndarray):
            if (
                image.ndim != 3
                or image.shape[-1] != 3
            ):
                raise ValueError(
                    f"Invalid image shape at index "
                    f"{index}: {image.shape}"
                )

            # OpenCV BGR -> RGB
            image = image[..., ::-1]

            image = np.ascontiguousarray(
                image,
                dtype=np.uint8
            )

            image = Image.fromarray(image)

        elif isinstance(image, str):
            image = Image.open(
                image
            ).convert("RGB")

        elif isinstance(image, Image.Image):
            image = image.convert("RGB")

        else:
            raise TypeError(
                f"Unsupported image type at index "
                f"{index}: {type(image)}"
            )

        image = self.transform(image)

        label = torch.tensor(
            label,
            dtype=torch.long
        )

        return {
            "pixel_values": image,
            "labels": label
        }
    #optimizer once
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=LEARNING_RATE)
    #Validation function

# ============================================================
# CROSSVIT VALIDATION AND TEST EVALUATION
# ============================================================

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    auc
)


def evaluate_crossvit(
    model,
    data_loader,
    criterion,
    device,
    return_details=False
):
    """
    Evaluate CrossViT.

    When return_details=False:
        Returns loss and accuracy for validation during training.

    When return_details=True:
        Returns complete test metrics, confusion matrix,
        classification report, and individual predictions.

    Labels:
        0 = real
        1 = fake
    """

    model.eval()

    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    all_labels = []
    all_predictions = []
    all_fake_probabilities = []

    with torch.inference_mode():

        for batch in data_loader:

            images = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            labels = batch[
                "labels"
            ].to(
                device,
                dtype=torch.long,
                non_blocking=True
            )

            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            )

            predictions = torch.argmax(
                logits,
                dim=1
            )

            fake_probabilities = probabilities[
                :,
                1
            ]

            current_batch_size = labels.size(0)

            total_loss += (
                loss.item()
                * current_batch_size
            )

            correct_predictions += (
                predictions == labels
            ).sum().item()

            total_samples += current_batch_size

            all_labels.extend(
                labels.detach().cpu().numpy()
            )

            all_predictions.extend(
                predictions.detach().cpu().numpy()
            )

            all_fake_probabilities.extend(
                fake_probabilities
                .detach()
                .cpu()
                .numpy()
            )

    if total_samples == 0:
        raise RuntimeError(
            "The supplied data loader contains no samples."
        )

    average_loss = (
        total_loss / total_samples
    )

    accuracy = (
        correct_predictions / total_samples
    )

    # Preserve compatibility with the training function
    if not return_details:
        return average_loss, accuracy

    # ========================================================
    # COMPLETE TEST METRICS
    # ========================================================

    y_true = np.asarray(
        all_labels,
        dtype=np.int64
    )

    y_pred = np.asarray(
        all_predictions,
        dtype=np.int64
    )

    y_score = np.asarray(
        all_fake_probabilities,
        dtype=np.float64
    )

    confusion = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = confusion.ravel()

    balanced_accuracy = balanced_accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    f1 = f1_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_true,
        y_pred
    )

    false_positive_rate = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else np.nan
    )

    false_negative_rate = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else np.nan
    )

    # ROC-AUC, PR-AUC, AP, and EER
    if len(np.unique(y_true)) == 2:

        roc_auc = roc_auc_score(
            y_true,
            y_score
        )

        average_precision = (
            average_precision_score(
                y_true,
                y_score
            )
        )

        pr_precision, pr_recall, _ = (
            precision_recall_curve(
                y_true,
                y_score
            )
        )

        pr_auc = auc(
            pr_recall,
            pr_precision
        )

        roc_fpr, roc_tpr, roc_thresholds = (
            roc_curve(
                y_true,
                y_score,
                pos_label=1
            )
        )

        roc_fnr = 1.0 - roc_tpr

        eer_index = np.nanargmin(
            np.abs(
                roc_fpr - roc_fnr
            )
        )

        eer = (
            roc_fpr[eer_index]
            + roc_fnr[eer_index]
        ) / 2.0

        eer_threshold = (
            roc_thresholds[eer_index]
        )

    else:
        roc_auc = np.nan
        pr_auc = np.nan
        average_precision = np.nan
        eer = np.nan
        eer_threshold = np.nan

    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=[
            "real",
            "fake"
        ],
        digits=4,
        zero_division=0
    )

    results = {
        "test_loss": average_loss,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall_sensitivity": recall,
        "specificity": specificity,
        "f1_score": f1,
        "mcc": mcc,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "average_precision": average_precision,
        "eer": eer,
        "eer_threshold": eer_threshold,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
        "number_of_test_images": int(total_samples)
    }

    predictions_df = pd.DataFrame({
        "true_label": y_true,
        "predicted_label": y_pred,
        "fake_probability": y_score
    })

    return (
        results,
        confusion,
        report,
        predictions_df
    )
def train_crossvit(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    epochs=10
):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": []
    }

    for epoch in range(epochs):
        model.train()

        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        for batch in train_loader:
            images = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            labels = batch[
                "labels"
            ].to(
                device,
                dtype=torch.long,
                non_blocking=True
            )

            optimizer.zero_grad()

            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

            loss.backward()
            optimizer.step()

            predictions = torch.argmax(
                logits,
                dim=1
            )

            current_batch_size = (
                labels.size(0)
            )

            running_loss += (
                loss.item()
                * current_batch_size
            )

            correct_predictions += (
                predictions == labels
            ).sum().item()

            total_samples += (
                current_batch_size
            )

        train_loss = (
            running_loss / total_samples
        )

        train_accuracy = (
            correct_predictions
            / total_samples
        )

        val_loss, val_accuracy = (
            evaluate_crossvit(
                model,
                val_loader,
                criterion,
                device
            )
        )

        history["train_loss"].append(
            train_loss
        )

        history["train_accuracy"].append(
            train_accuracy
        )

        history["val_loss"].append(
            val_loss
        )

        history["val_accuracy"].append(
            val_accuracy
        )

        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"Train loss: {train_loss:.4f} | "
            f"Train accuracy: {train_accuracy:.4f} | "
            f"Validation loss: {val_loss:.4f} | "
            f"Validation accuracy: {val_accuracy:.4f}"
        )

    return history

Device: cuda
Loaded model: crossvit_15_240.in1k
Branch 0: (240, 240) -> (160, 160) | grid (20, 20) -> (13, 13)
Branch 1: (224, 224) -> (149, 149) | grid (14, 14) -> (9, 9)

CrossViT adapted successfully.
Trainable parameters: 26863108
Total parameters: 26863108
Test output shape: torch.Size([2, 2])


# Wild deepfake

In [20]:
import h5py
import numpy as np

H5_PATH = (
    r"D:\thesis\dataset\WildDeepfake\leakage_free_subset"
    r"\wilddeepfake_sequence_disjoint_face_preprocessed.h5"
)

with h5py.File(H5_PATH, "r") as h5f:
    # Load image arrays
    train_images = h5f["train_images"][:]
    train_labels = h5f["train_labels"][:]

    val_images = h5f["val_images"][:]
    val_labels = h5f["val_labels"][:]

    test_images = h5f["test_images"][:]
    test_labels = h5f["test_labels"][:]

# Verify dataset sizes
print(f"Total train: {len(train_images)} images")
print(f"Total validation: {len(val_images)} images")
print(f"Total test: {len(test_images)} images")

print(f"Train labels: {len(train_labels)}")
print(f"Validation labels: {len(val_labels)}")
print(f"Test labels: {len(test_labels)}")

# Verify shapes and data types
print("\nArray information:")
print(f"Train images: {train_images.shape}, dtype={train_images.dtype}")
print(f"Validation images: {val_images.shape}, dtype={val_images.dtype}")
print(f"Test images: {test_images.shape}, dtype={test_images.dtype}")

print(f"Train labels: {train_labels.shape}, dtype={train_labels.dtype}")
print(f"Validation labels: {val_labels.shape}, dtype={val_labels.dtype}")
print(f"Test labels: {test_labels.shape}, dtype={test_labels.dtype}")

# Verify class distributions
print("\nClass distribution:")
print(
    f"Train: Real={np.sum(train_labels == 0)}, "
    f"Fake={np.sum(train_labels == 1)}"
)
print(
    f"Validation: Real={np.sum(val_labels == 0)}, "
    f"Fake={np.sum(val_labels == 1)}"
)
print(
    f"Test: Real={np.sum(test_labels == 0)}, "
    f"Fake={np.sum(test_labels == 1)}"
)

Total train: 36000 images
Total validation: 6000 images
Total test: 18000 images
Train labels: 36000
Validation labels: 6000
Test labels: 18000

Array information:
Train images: (36000, 160, 160, 3), dtype=uint8
Validation images: (6000, 160, 160, 3), dtype=uint8
Test images: (18000, 160, 160, 3), dtype=uint8
Train labels: (36000,), dtype=uint8
Validation labels: (6000,), dtype=uint8
Test labels: (18000,), dtype=uint8

Class distribution:
Train: Real=9000, Fake=27000
Validation: Real=1500, Fake=4500
Test: Real=4500, Fake=13500


In [7]:
# ============================================================
# DATASETS AND DATALOADERS
# ============================================================

train_dataset = DeepfakeCrossViTDataset(
    train_images,
    train_labels,
    crossvit_transform
)

val_dataset = DeepfakeCrossViTDataset(
    val_images,
    val_labels,
    crossvit_transform
)

test_dataset = DeepfakeCrossViTDataset(
    test_images,
    test_labels,
    crossvit_transform
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training samples: 36000
Validation samples: 6000
Testing samples: 18000
Training batches: 2250
Validation batches: 375
Testing batches: 1125


In [8]:
history = train_crossvit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=EPOCHS
)

Epoch 01/10 | Train loss: 0.2160 | Train accuracy: 0.9121 | Validation loss: 0.1814 | Validation accuracy: 0.9320
Epoch 02/10 | Train loss: 0.0663 | Train accuracy: 0.9756 | Validation loss: 0.1981 | Validation accuracy: 0.9390
Epoch 03/10 | Train loss: 0.0423 | Train accuracy: 0.9849 | Validation loss: 0.2447 | Validation accuracy: 0.9257
Epoch 04/10 | Train loss: 0.0330 | Train accuracy: 0.9885 | Validation loss: 0.2180 | Validation accuracy: 0.9367
Epoch 05/10 | Train loss: 0.0265 | Train accuracy: 0.9906 | Validation loss: 0.2508 | Validation accuracy: 0.9335
Epoch 06/10 | Train loss: 0.0240 | Train accuracy: 0.9913 | Validation loss: 0.1965 | Validation accuracy: 0.9382
Epoch 07/10 | Train loss: 0.0194 | Train accuracy: 0.9931 | Validation loss: 0.2364 | Validation accuracy: 0.9435
Epoch 08/10 | Train loss: 0.0186 | Train accuracy: 0.9941 | Validation loss: 0.2345 | Validation accuracy: 0.9442
Epoch 09/10 | Train loss: 0.0175 | Train accuracy: 0.9941 | Validation loss: 0.2634 | Va

In [9]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [10]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [21]:
test_results, confusion, classification_report_text, predictions_df = (
    evaluate_crossvit(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)

In [ ]:
print("\nCROSSVIT TEST RESULTS")
print("=" * 60)

for metric, value in test_results.items():
    if isinstance(value, (float, np.floating)):
        print(f"{metric:30s}: {value:.6f}")
    else:
        print(f"{metric:30s}: {value}")
print("\nConfusion Matrix:")
print(confusion)

print("\nFormat:")
print("[[TN, FP],")
print(" [FN, TP]]")
print("\nClassification Report:")
print(classification_report_text)

CROSSVIT TEST RESULTS
test_loss                     : 0.944846
accuracy                      : 0.8004
balanced_accuracy             : 0.8087
precision                     : 0.9315
recall_sensitivity            : 0.7921
specificity                   : 0.8253
f1_score                      : 0.8565
mcc                           : 0.5684
roc_auc                       : 0.8863
pr_auc                        : 0.9582
average_precision             : 0.9582
eer                           : 0.1949
eer_threshold                 : 0.851565
false_positive_rate           : 0.1747
false_negative_rate           : 0.2079
true_negatives                : 3714
false_positives               : 786
false_negatives               : 2807
true_positives                : 10693
number_of_test_images         : 18000

Confusion Matrix:
[[ 3714   786]
 [ 2807 10693]]

Format:
[[TN, FP],
 [FN, TP]]

Classification Report:
              precision    recall  f1-score   support

        real     0.5696    0.8253    0.6741

In [12]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
#print(f"RAM Used: {end['ram_mb'] - start['ram_mb']:.1f} MB")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 17.5%
Time Usage: 146.2 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [13]:
end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 13.2%
Time Usage: 146.4 s
GPU Memory Used: 424.3 MB
Power Consumption: 93W


save the model

In [14]:
SAVE_DIR = (
    r"D:\thesis\results"
    r"\crossvit_15_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

CHECKPOINT_PATH = os.path.join(
    SAVE_DIR,
    "crossvit_15_checkpoint.pt"
)

torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "history":
            history,

        "model_name":
            MODEL_NAME,

        "input_size":
            INPUT_SIZE,

        "epochs":
            EPOCHS,

        "batch_size":
            BATCH_SIZE,

        "learning_rate":
            LEARNING_RATE,

        "num_classes":
            2,

        "label_mapping": {
            0: "real",
            1: "fake"
        },

        "position_embeddings_resized":
            True,

        "original_input_size":
            240
    },
    CHECKPOINT_PATH
)

print("CrossViT checkpoint saved:")
print(CHECKPOINT_PATH)

CrossViT checkpoint saved:
D:\thesis\results\crossvit_15_160\crossvit_15_checkpoint.pt


load the model

In [ ]:
import os
import torch

from transformers import (
    ViTImageProcessor,
    ViTForImageClassification
)


SAVE_DIR = (
    r"D:\thesis\results"
    r"\crossvit_15_160"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Model directory exists:", os.path.isdir(SAVE_DIR))


# Load the saved processor
processor = ViTImageProcessor.from_pretrained(
    SAVE_DIR
)

# Load the saved model configuration and final trained weights
model = ViTForImageClassification.from_pretrained(
    SAVE_DIR
)

# Move the model to GPU or CPU
model = model.to(device)

# Inference mode
model.eval()


print("Cross-ViR model loaded successfully")
print("Number of classes:", model.config.num_labels)
print("Label mapping:", model.config.id2label)

print(
    "Total parameters:",
    sum(parameter.numel() for parameter in model.parameters())
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)

Device: cuda
Model directory exists: True
ViT-16 model loaded successfully
Number of classes: 2
Label mapping: {0: 'real', 1: 'fake'}
Total parameters: 87456770
Trainable parameters: 87456770


In [15]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_4256\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [17]:
# ============================================================
# PYTORCH CROSSVIT WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# ============================================================
# DEVICE AND MODEL
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(
    test_loader.dataset
)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ============================================================
# GPU/MODEL WARM-UP
# ============================================================

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():

    for batch_index, batch in enumerate(
        test_loader
    ):

        if batch_index >= WARMUP_BATCHES:
            break

        images = batch[
            "pixel_values"
        ].to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        # timm CrossViT accepts the image tensor directly
        logits = model(images)

        # Materialize the output
        _ = logits


# Wait until all asynchronous GPU operations finish
if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ============================================================
# FIVE REPEATED INFERENCE RUNS
# ============================================================

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):

    print(
        f"\nStarting CrossViT resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous GPU work remains queued
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():

        for batch in test_loader:

            images = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            # CrossViT returns the logits tensor directly
            logits = model(images)

            last_logits = logits

            processed_images += images.size(0)


    # Ensure inference is complete before stopping timing
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )


    # Verify that at least one batch was processed
    if last_logits is not None:

        last_output_value = float(
            last_logits[-1, 0]
            .detach()
            .cpu()
            .item()
        )

    else:
        raise RuntimeError(
            "No images were processed during inference."
        )


    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number
    run_summary[
        "processed_images"
    ] = processed_images

    run_summary[
        "last_output_value"
    ] = last_output_value

    run_results.append(
        run_summary
    )


    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )


    del last_logits
    del logits


results_df = pd.DataFrame(
    run_results
)

print(
    "\nIndividual CrossViT profiling runs:"
)

display(results_df)

Device: cuda
Test images: 18000
Test batches: 1125

Performing warm-up using 3 batches...
Warm-up completed.

Starting CrossViT resource run 1/5
Run 1: 28.97 seconds | 1.6096 ms/image | 621.26 images/s

Starting CrossViT resource run 2/5
Run 2: 24.67 seconds | 1.3705 ms/image | 729.64 images/s

Starting CrossViT resource run 3/5
Run 3: 23.94 seconds | 1.3299 ms/image | 751.91 images/s

Starting CrossViT resource run 4/5
Run 4: 51.61 seconds | 2.8672 ms/image | 348.77 images/s

Starting CrossViT resource run 5/5
Run 5: 37.62 seconds | 2.0901 ms/image | 478.45 images/s

Individual CrossViT profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,28.973517,1.609640,621.256989,5.081238,14.700000,1021.878906,1678.948312,2286.292969,657.069405,1264.414062,...,0.0,0.0,69.988764,96,52.260592,75.435,0.422685,1,18000,1.731751
1,24.669857,1.370548,729.635360,4.927583,12.990625,2286.957031,2287.450694,2291.792969,0.493663,4.835938,...,0.0,0.0,86.977778,100,53.614942,116.900,0.370141,2,18000,1.731751
2,23.939051,1.329947,751.909506,5.252955,15.093750,2287.046875,2287.690909,2292.152344,0.644034,5.105469,...,0.0,0.0,91.900000,100,53.394064,97.469,0.357574,3,18000,1.731751
3,51.610222,2.867235,348.768115,4.268450,12.990625,2287.386719,2287.891258,2292.472656,0.504539,5.085938,...,0.0,0.0,43.368870,100,38.396497,120.998,0.552379,4,18000,1.731751
4,37.621313,2.090073,478.452199,4.626715,12.543750,2287.859375,2288.230757,2292.792969,0.371382,4.933594,...,0.0,0.0,50.902655,100,54.460478,117.078,0.574042,5,18000,1.731751


In [20]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("Cross-ViT RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


Cross-ViT RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,33.362792,11.562260,19.006350,47.719235
1,latency_ms_per_image,1.853488,0.642348,1.055908,2.651069
2,throughput_images_per_s,586.004434,171.205748,373.424420,798.584448
3,average_cpu_percent,4.831388,0.389932,4.347224,5.315552
4,peak_cpu_percent,13.663750,1.148837,12.237280,15.090220
5,average_ram_mb,2166.042386,272.294015,1827.944710,2504.140062
6,peak_ram_mb,2291.100781,2.713186,2287.731916,2294.469646
7,average_incremental_ram_mb,131.816605,293.625258,-232.767307,496.400516
8,peak_incremental_ram_mb,256.875000,563.231469,-442.469238,956.219238
9,average_gpu_memory_mb,2723.429688,0.000000,2723.429688,2723.429688



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 1.853 ± 0.642 (95% CI: 1.056–2.651)
peak_ram_mb: 2291.101 ± 2.713 (95% CI: 2287.732–2294.470)
peak_gpu_memory_mb: 2723.430 ± 0.000 (95% CI: 2723.430–2723.430)
average_gpu_utilization_percent: 68.628 ± 21.403 (95% CI: 42.052–95.203)
average_gpu_power_w: 50.425 ± 6.770 (95% CI: 42.019–58.831)


genralization

In [24]:
print("\nTest results of wild deepfake dataset on Celeb-DF(V2) (Cross-ViT):")
test_dataset = DeepfakeCrossViTDataset(test_celeb,test_labels,crossvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, classification_report_text, predictions_df = (
    evaluate_crossvit(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)
for metric, value in test_results.items():
    if isinstance(value, (float, np.floating)):
        print(f"{metric:30s}: {value:.6f}")
    else:
        print(f"{metric:30s}: {value}")


Test results of wild deepfake dataset on Celeb-DF(V2) (Cross-ViT):
test_loss                     : 1.883394
accuracy                      : 0.538165
balanced_accuracy             : 0.628752
precision                     : 0.949540
recall_sensitivity            : 0.516698
specificity                   : 0.740806
f1_score                      : 0.669230
mcc                           : 0.151587
roc_auc                       : 0.688248
pr_auc                        : 0.949844
average_precision             : 0.949859
eer                           : 0.365253
eer_threshold                 : 0.153999
false_positive_rate           : 0.259194
false_negative_rate           : 0.483302
true_negatives                : 423
false_positives               : 148
false_negatives               : 2605
true_positives                : 2785
number_of_test_images         : 5961


In [26]:
#dfc on wilddeepfake
print("\nTest results of wild deepfake dataset on DFC (Cross-ViT):")
test_dataset = DeepfakeCrossViTDataset(test_hog,test_labels,crossvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, classification_report_text, predictions_df = (
    evaluate_crossvit(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)
for metric, value in test_results.items():
    if isinstance(value, (float, np.floating)):
        print(f"{metric:30s}: {value:.6f}")
    else:
        print(f"{metric:30s}: {value}")



Test results of wild deepfake dataset on DFC (Cross-ViT):
test_loss                     : 2.127741
accuracy                      : 0.525333
balanced_accuracy             : 0.525333
precision                     : 0.548969
recall_sensitivity            : 0.284000
specificity                   : 0.766667
f1_score                      : 0.374341
mcc                           : 0.057852
roc_auc                       : 0.530660
pr_auc                        : 0.526773
average_precision             : 0.527923
eer                           : 0.477000
eer_threshold                 : 0.046291
false_positive_rate           : 0.233333
false_negative_rate           : 0.716000
true_negatives                : 1150
false_positives               : 350
false_negatives               : 1074
true_positives                : 426
number_of_test_images         : 3000


In [28]:
print("\nTest results of wild deepfake dataset on FF++ (Cross-ViT):")
#ff++ on wilddeepfake
test_dataset = DeepfakeCrossViTDataset(test_ff,test_ff_labels,crossvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, classification_report_text, predictions_df = (
    evaluate_crossvit(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)
for metric, value in test_results.items():
    if isinstance(value, (float, np.floating)):
        print(f"{metric:30s}: {value:.6f}")
    else:
        print(f"{metric:30s}: {value}")


Test results of wild deepfake dataset on FF++ (Cross-ViT):
test_loss                     : 1.894187
accuracy                      : 0.566869
balanced_accuracy             : 0.541008
precision                     : 0.475508
recall_sensitivity            : 0.386033
specificity                   : 0.695983
f1_score                      : 0.426124
mcc                           : 0.085466
roc_auc                       : 0.557387
pr_auc                        : 0.477297
average_precision             : 0.477929
eer                           : 0.452102
eer_threshold                 : 0.092847
false_positive_rate           : 0.304017
false_negative_rate           : 0.613967
true_negatives                : 1005
false_positives               : 439
false_negatives               : 633
true_positives                : 398
number_of_test_images         : 2475


# Celeb

In [23]:
import os
import cv2
import numpy as np

SAVE_ROOT = r'D:\thesis\celeb_processed'

def load_split(split_name, class_name):
    """Reload saved frames, grouped by video."""
    base = os.path.join(SAVE_ROOT, split_name, class_name)
    nested, ids = [], []
    for vid_id in sorted(os.listdir(base)):
        vid_dir = os.path.join(base, vid_id)
        frames = [cv2.imread(os.path.join(vid_dir, f))
                  for f in sorted(os.listdir(vid_dir))]
        if frames:
            nested.append(frames)
            ids.append(vid_id)
    return nested, ids

# Reload ALL six splits
print("Loading frames...")
real_train_final,  real_train_ids  = load_split('train', 'real')
synth_train_final, synth_train_ids = load_split('train', 'fake')
real_val_final,    real_val_ids    = load_split('val',   'real')
synth_val_final,   synth_val_ids   = load_split('val',   'fake')
real_test_final,   real_test_ids   = load_split('test',  'real')
synth_test_final,  synth_test_ids  = load_split('test',  'fake')

print("✅ All frames reloaded")
print("Train -> real videos:", len(real_train_final), " fake videos:", len(synth_train_final))
print("Val   -> real videos:", len(real_val_final),   " fake videos:", len(synth_val_final))
print("Test  -> real videos:", len(real_test_final),  " fake videos:", len(synth_test_final))
print("Example frame shape:", np.shape(real_train_final[0][0]))  # expect (160,160,3)
import numpy as np


def combine_split(real_videos, fake_videos):
    """
    Flatten video-grouped frames into one image array and create labels.

    real_videos: list of videos, where each video is a list of frames
    fake_videos: list of videos, where each video is a list of frames

    Returns
    -------
    images : NumPy array with shape (N, 160, 160, 3)
    labels : NumPy array with shape (N,)
             0 = real, 1 = fake
    """

    # Flatten frames from all real videos
    real_frames = [
        frame
        for video_frames in real_videos
        for frame in video_frames
        if frame is not None
    ]

    # Flatten frames from all fake videos
    fake_frames = [
        frame
        for video_frames in fake_videos
        for frame in video_frames
        if frame is not None
    ]

    if len(real_frames) == 0:
        raise ValueError("No real frames were found.")

    if len(fake_frames) == 0:
        raise ValueError("No fake frames were found.")

    # Convert to NumPy arrays
    real_frames = np.stack(real_frames).astype(np.uint8)
    fake_frames = np.stack(fake_frames).astype(np.uint8)

    # Combine images
    images = np.concatenate(
        [real_frames, fake_frames],
        axis=0
    )

    # Create labels
    real_labels = np.zeros(
        len(real_frames),
        dtype=np.uint8
    )

    fake_labels = np.ones(
        len(fake_frames),
        dtype=np.uint8
    )

    labels = np.concatenate(
        [real_labels, fake_labels],
        axis=0
    )

    return images, labels
# Training set
train_celeb, train_labels = combine_split(
    real_train_final,
    synth_train_final
)

# Validation set
val_celeb, val_labels = combine_split(
    real_val_final,
    synth_val_final
)

# Testing set
test_celeb, test_labels = combine_split(
    real_test_final,
    synth_test_final
)
print("\nTRAIN")
print("Images:", train_celeb.shape)
print("Labels:", train_labels.shape)
print("Real:", np.sum(train_labels == 0))
print("Fake:", np.sum(train_labels == 1))

print("\nVALIDATION")
print("Images:", val_celeb.shape)
print("Labels:", val_labels.shape)
print("Real:", np.sum(val_labels == 0))
print("Fake:", np.sum(val_labels == 1))

print("\nTEST")
print("Images:", test_celeb.shape)
print("Labels:", test_labels.shape)
print("Real:", np.sum(test_labels == 0))
print("Fake:", np.sum(test_labels == 1))

print("\nData types")
print("Train images:", train_celeb.dtype)
print("Train labels:", train_labels.dtype)

Loading frames...
✅ All frames reloaded
Train -> real videos: 354  fake videos: 3383
Val   -> real videos: 59  fake videos: 563
Test  -> real videos: 177  fake videos: 1693
Example frame shape: (160, 160, 3)

TRAIN
Images: (11899, 160, 160, 3)
Labels: (11899,)
Real: 1142
Fake: 10757

VALIDATION
Images: (1969, 160, 160, 3)
Labels: (1969,)
Real: 182
Fake: 1787

TEST
Images: (5961, 160, 160, 3)
Labels: (5961,)
Real: 571
Fake: 5390

Data types
Train images: uint8
Train labels: uint8


In [7]:
# ============================================================
# DATASETS AND DATALOADERS
# ============================================================

train_dataset = DeepfakeCrossViTDataset(
    train_celeb,
    train_labels,
    crossvit_transform
)

val_dataset = DeepfakeCrossViTDataset(
    val_celeb,
    val_labels,
    crossvit_transform
)

test_dataset = DeepfakeCrossViTDataset(
    test_celeb,
    test_labels,
    crossvit_transform
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training samples: 11899
Validation samples: 1969
Testing samples: 5961
Training batches: 744
Validation batches: 124
Testing batches: 373


In [8]:
history = train_crossvit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=EPOCHS
)

Epoch 01/10 | Train loss: 0.2847 | Train accuracy: 0.9044 | Validation loss: 0.2763 | Validation accuracy: 0.9101
Epoch 02/10 | Train loss: 0.1695 | Train accuracy: 0.9338 | Validation loss: 0.1911 | Validation accuracy: 0.9319
Epoch 03/10 | Train loss: 0.0829 | Train accuracy: 0.9690 | Validation loss: 0.1619 | Validation accuracy: 0.9406
Epoch 04/10 | Train loss: 0.0572 | Train accuracy: 0.9792 | Validation loss: 0.2035 | Validation accuracy: 0.9396
Epoch 05/10 | Train loss: 0.0410 | Train accuracy: 0.9861 | Validation loss: 0.1538 | Validation accuracy: 0.9538
Epoch 06/10 | Train loss: 0.0280 | Train accuracy: 0.9905 | Validation loss: 0.2306 | Validation accuracy: 0.9370
Epoch 07/10 | Train loss: 0.0381 | Train accuracy: 0.9880 | Validation loss: 0.1575 | Validation accuracy: 0.9482
Epoch 08/10 | Train loss: 0.0243 | Train accuracy: 0.9917 | Validation loss: 0.1752 | Validation accuracy: 0.9538
Epoch 09/10 | Train loss: 0.0274 | Train accuracy: 0.9908 | Validation loss: 0.2942 | Va

In [9]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [10]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [ ]:
test_results, confusion, classification_report_text, predictions_df = (
    evaluate_crossvit(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)
for metric, value in test_results.items():
    if isinstance(value, (float, np.floating)):
        print(f"{metric:30s}: {value:.6f}")
    else:
        print(f"{metric:30s}: {value}")
print("\nConfusion Matrix:")
print(confusion)

print("\nFormat:")
print("[[TN, FP],")
print(" [FN, TP]]")
print("\nClassification Report:")
print(classification_report_text)


test_loss                     : 0.256659
accuracy                      : 0.957390
balanced_accuracy             : 0.783063
precision                     : 0.956128
recall_sensitivity            : 0.998701
specificity                   : 0.567426
f1_score                      : 0.976951
mcc                           : 0.727542
roc_auc                       : 0.984539
pr_auc                        : 0.998116
average_precision             : 0.998060
eer                           : 0.054604
eer_threshold                 : 0.999956
false_positive_rate           : 0.432574
false_negative_rate           : 0.001299
true_negatives                : 324
false_positives               : 247
false_negatives               : 7
true_positives                : 5383
number_of_test_images         : 5961

Confusion Matrix:
[[ 324  247]
 [   7 5383]]

Confusion Matrix Format:
[[TN, FP],
 [FN, TP]]

Classification Report
              precision    recall  f1-score   support

        Real   0.978852  0.56742

In [12]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 8.1%
Time Usage: 14.2 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [13]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 7.5%
Time Usage: 16.3 s
GPU Memory Used: 424.3 MB
Power Consumption: 93W


save the model

In [14]:
SAVE_DIR = (
    r"D:\thesis\results"
    r"\crossvit_15_160_celeb"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

CHECKPOINT_PATH = os.path.join(
    SAVE_DIR,
    "crossvit_15_checkpoint_celeb.pt"
)

torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "history":
            history,

        "model_name":
            MODEL_NAME,

        "input_size":
            INPUT_SIZE,

        "epochs":
            EPOCHS,

        "batch_size":
            BATCH_SIZE,

        "learning_rate":
            LEARNING_RATE,

        "num_classes":
            2,

        "label_mapping": {
            0: "real",
            1: "fake"
        },

        "position_embeddings_resized":
            True,

        "original_input_size":
            240
    },
    CHECKPOINT_PATH
)

print("CrossViT checkpoint saved:")
print(CHECKPOINT_PATH)

CrossViT checkpoint saved:
D:\thesis\results\crossvit_15_160_celeb\crossvit_15_checkpoint_celeb.pt


load the model

In [ ]:
import os
import torch

from transformers import (
    ViTImageProcessor,
    ViTForImageClassification
)


SAVE_DIR = (
    r"D:\thesis\results"
    r"\vit_base_celeb_patch32_160"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Model directory exists:", os.path.isdir(SAVE_DIR))


# Load the saved processor
processor = ViTImageProcessor.from_pretrained(
    SAVE_DIR
)

# Load the saved model configuration and final trained weights
model = ViTForImageClassification.from_pretrained(
    SAVE_DIR
)

# Move the model to GPU or CPU
model = model.to(device)

# Inference mode
model.eval()


print("ViT-16 model loaded successfully")
print("Number of classes:", model.config.num_labels)
print("Label mapping:", model.config.id2label)

print(
    "Total parameters:",
    sum(parameter.numel() for parameter in model.parameters())
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)

In [15]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_32800\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [16]:
# ============================================================
# PYTORCH CROSSVIT WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# ============================================================
# DEVICE AND MODEL
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(
    test_loader.dataset
)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ============================================================
# GPU/MODEL WARM-UP
# ============================================================

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():

    for batch_index, batch in enumerate(
        test_loader
    ):

        if batch_index >= WARMUP_BATCHES:
            break

        images = batch[
            "pixel_values"
        ].to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        # timm CrossViT accepts the image tensor directly
        logits = model(images)

        # Materialize the output
        _ = logits


# Wait until all asynchronous GPU operations finish
if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ============================================================
# FIVE REPEATED INFERENCE RUNS
# ============================================================

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):

    print(
        f"\nStarting CrossViT resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous GPU work remains queued
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():

        for batch in test_loader:

            images = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            # CrossViT returns the logits tensor directly
            logits = model(images)

            last_logits = logits

            processed_images += images.size(0)


    # Ensure inference is complete before stopping timing
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )


    # Verify that at least one batch was processed
    if last_logits is not None:

        last_output_value = float(
            last_logits[-1, 0]
            .detach()
            .cpu()
            .item()
        )

    else:
        raise RuntimeError(
            "No images were processed during inference."
        )


    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number
    run_summary[
        "processed_images"
    ] = processed_images

    run_summary[
        "last_output_value"
    ] = last_output_value

    run_results.append(
        run_summary
    )


    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )


    del last_logits
    del logits


results_df = pd.DataFrame(
    run_results
)

print(
    "\nIndividual CrossViT profiling runs:"
)

display(results_df)

Device: cuda
Test images: 5961
Test batches: 373

Performing warm-up using 3 batches...
Warm-up completed.

Starting CrossViT resource run 1/5
Run 1: 10.28 seconds | 1.7242 ms/image | 579.99 images/s

Starting CrossViT resource run 2/5
Run 2: 9.41 seconds | 1.5782 ms/image | 633.63 images/s

Starting CrossViT resource run 3/5
Run 3: 9.39 seconds | 1.5756 ms/image | 634.67 images/s

Starting CrossViT resource run 4/5
Run 4: 9.56 seconds | 1.6039 ms/image | 623.48 images/s

Starting CrossViT resource run 5/5
Run 5: 10.31 seconds | 1.7294 ms/image | 578.22 images/s

Individual CrossViT profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,10.277845,1.724181,579.985373,4.757270,12.543750,7523.539062,7524.155263,7528.519531,0.616201,4.980469,...,11.747368,12.0,62.684211,93,52.226379,104.051,0.151279,1,5961,-2.958673
1,9.407702,1.578209,633.629757,4.794540,9.375000,7523.925781,7524.260282,7528.804688,0.334501,4.878906,...,0.000000,0.0,68.379310,99,55.708793,89.639,0.147899,2,5961,-2.958673
2,9.392247,1.575616,634.672413,4.810668,11.096875,7524.042969,7524.460578,7528.847656,0.417610,4.804688,...,11.724138,12.0,68.321839,100,54.726299,98.479,0.144757,3,5961,-2.958673
3,9.560822,1.603896,623.481975,4.606706,12.543750,7524.058594,7524.819917,7529.160156,0.761324,5.101562,...,0.000000,0.0,67.966292,100,54.337191,116.988,0.147527,4,5961,-2.958673
4,10.309238,1.729448,578.219274,4.667270,10.750000,7524.367188,7524.674383,7528.609375,0.307196,4.242188,...,11.747368,12.0,59.852632,100,57.234158,98.984,0.166391,5,5961,-2.958673


In [17]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("Cross-ViT RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


Cross-ViT RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,9.789571,0.464888,9.212337,10.366805
1,latency_ms_per_image,1.642270,0.077988,1.545435,1.739105
2,throughput_images_per_s,609.997759,28.547107,574.551843,645.443674
3,average_cpu_percent,4.727291,0.087344,4.618839,4.835743
4,peak_cpu_percent,11.261875,1.335660,9.603435,12.920315
5,average_ram_mb,7524.474085,0.277156,7524.129951,7524.818219
6,peak_ram_mb,7528.788281,0.248153,7528.480159,7529.096404
7,average_incremental_ram_mb,0.487366,0.195158,0.245046,0.729687
8,peak_incremental_ram_mb,4.801563,0.332001,4.389328,5.213797
9,average_gpu_memory_mb,2615.101587,0.142928,2614.924119,2615.279056



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 1.642 ± 0.078 (95% CI: 1.545–1.739)
peak_ram_mb: 7528.788 ± 0.248 (95% CI: 7528.480–7529.096)
peak_gpu_memory_mb: 2615.258 ± 0.000 (95% CI: 2615.258–2615.258)
average_gpu_utilization_percent: 65.441 ± 3.941 (95% CI: 60.547–70.335)
average_gpu_power_w: 54.847 ± 1.843 (95% CI: 52.558–57.135)


#genralization

In [19]:
#wild deepfake on celeb
print("\nTest results of Celeb-DF(V2) on wild deepfake dataset (Cross-ViT):")
test_dataset = DeepfakeCrossViTDataset(test_images,test_labels,crossvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, classification_report_text, predictions_df = (
    evaluate_crossvit(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)
for metric, value in test_results.items():
    if isinstance(value, (float, np.floating)):
        print(f"{metric:30s}: {value:.6f}")
    else:
        print(f"{metric:30s}: {value}")



Test results of Celeb-DF(V2) on wild deepfake dataset (Cross-ViT):
test_loss                     : 1.357078
accuracy                      : 0.562722
balanced_accuracy             : 0.595074
precision                     : 0.823841
recall_sensitivity            : 0.530370
specificity                   : 0.659778
f1_score                      : 0.645307
mcc                           : 0.164770
roc_auc                       : 0.623612
pr_auc                        : 0.842692
average_precision             : 0.842704
eer                           : 0.412185
eer_threshold                 : 0.326750
false_positive_rate           : 0.340222
false_negative_rate           : 0.469630
true_negatives                : 2969
false_positives               : 1531
false_negatives               : 6340
true_positives                : 7160
number_of_test_images         : 18000


In [21]:
#DFC on celeb
print("\nTest results of Celeb-DF(V2) on DFC dataset (Cross-ViT):")
test_dataset = DeepfakeCrossViTDataset(test_hog,test_labels,crossvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, classification_report_text, predictions_df = (
    evaluate_crossvit(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)
for metric, value in test_results.items():
    if isinstance(value, (float, np.floating)):
        print(f"{metric:30s}: {value:.6f}")
    else:
        print(f"{metric:30s}: {value}")



Test results of Celeb-DF(V2) on DFC dataset (Cross-ViT):
test_loss                     : 1.754351
accuracy                      : 0.481000
balanced_accuracy             : 0.481000
precision                     : 0.488010
recall_sensitivity            : 0.773333
specificity                   : 0.188667
f1_score                      : 0.598401
mcc                           : -0.046840
roc_auc                       : 0.459034
pr_auc                        : 0.470581
average_precision             : 0.471550
eer                           : 0.529333
eer_threshold                 : 0.923890
false_positive_rate           : 0.811333
false_negative_rate           : 0.226667
true_negatives                : 283
false_positives               : 1217
false_negatives               : 340
true_positives                : 1160
number_of_test_images         : 3000


In [23]:
#FF++ on celeb
print("\nTest results of Celeb-DF(V2) on FF++ dataset (Cross-ViT):")
test_dataset = DeepfakeCrossViTDataset(test_ff,test_ff_labels,crossvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, classification_report_text, predictions_df = (
    evaluate_crossvit(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)
for metric, value in test_results.items():
    if isinstance(value, (float, np.floating)):
        print(f"{metric:30s}: {value:.6f}")
    else:
        print(f"{metric:30s}: {value}")


Test results of Celeb-DF(V2) on FF++ dataset (Cross-ViT):
test_loss                     : 1.119228
accuracy                      : 0.672323
balanced_accuracy             : 0.701151
precision                     : 0.569532
recall_sensitivity            : 0.873909
specificity                   : 0.528393
f1_score                      : 0.689629
mcc                           : 0.412987
roc_auc                       : 0.826689
pr_auc                        : 0.785726
average_precision             : 0.785965
eer                           : 0.254485
eer_threshold                 : 0.940906
false_positive_rate           : 0.471607
false_negative_rate           : 0.126091
true_negatives                : 763
false_positives               : 681
false_negatives               : 130
true_positives                : 901
number_of_test_images         : 2475


# DFC

In [25]:
import h5py
import numpy as np
# Open the HDF5 file in read mode
with h5py.File('D://thesis//dataset//deepfake dataset//resized_images.h5', 'r') as h5f:
    # Access each dataset
    celeb = np.array(h5f['celeb'])
    ffhq = np.array(h5f['ffhq'])
    gdwct = np.array(h5f['gdwct'])
    attgan = np.array(h5f['attgan'])
    stargan = np.array(h5f['stargan'])
    stylegan2 = np.array(h5f['stylegan2'])
    stylegan = np.array(h5f['stylegan'])

# Now, 'celeb', 'ffhq', etc., are NumPy arrays containing your datasets
print(f"celeb shape: {celeb.shape}, dtype: {celeb.dtype}")
print(f"ffhq shape: {ffhq.shape}, dtype: {ffhq.dtype}")
print(f"ffhq shape: {gdwct.shape}, dtype: {gdwct.dtype}")
print(f"ffhq shape: {attgan.shape}, dtype: {attgan.dtype}")
print(f"ffhq shape: {stargan.shape}, dtype: {stargan.dtype}")
print(f"ffhq shape: {stylegan2.shape}, dtype: {stylegan2.dtype}")
print(f"ffhq shape: {stylegan.shape}, dtype: {stylegan.dtype}")
# Repeat for other datasets as needed
import cv2
# Function to resize images from (224, 224) to (160, 160)
def resize_images(image_array, target_size=(160, 160)):
    resized_images = np.array([cv2.resize(img, target_size) for img in image_array])
    return resized_images

celeb = resize_images(celeb, target_size=(160, 160))
ffhq = resize_images(ffhq, target_size=(160, 160))
gdwct = resize_images(gdwct, target_size=(160, 160))
attgan = resize_images(attgan, target_size=(160, 160))
stargan = resize_images(stargan, target_size=(160, 160))
stylegan = resize_images(stylegan, target_size=(160, 160))
stylegan2 = resize_images(stylegan2, target_size=(160, 160))
import random
# Randomly select 2500 distinct images
random_indices = random.sample(range(len(celeb)), 2500)  # Get 2500 random indices
celeb = celeb[random_indices]  # Select the random subse

import random
# Randomly select 2500 distinct images
random_indices = random.sample(range(len(ffhq)), 2500)  # Get 2500 random indices
ffhq = ffhq[random_indices]  # Select the random subse
print(f"celeb shape: {celeb.shape}, dtype: {celeb.dtype}")
print(f"ffhq shape: {ffhq.shape}, dtype: {ffhq.dtype}")
print(f"gdwct shape: {gdwct.shape}, dtype: {gdwct.dtype}")
print(f"attagan shape: {attgan.shape}, dtype: {attgan.dtype}")
print(f"stargan shape: {stargan.shape}, dtype: {stargan.dtype}")
print(f"stylegan2 shape: {stylegan2.shape}, dtype: {stylegan2.dtype}")
print(f"stylegan shape: {stylegan.shape}, dtype: {stylegan.dtype}")
import random
import numpy as np

def split_data(data, train_ratio=0.7):
    """
    Splits data into training and testing sets based on the specified ratio.

    Parameters:
        data (list or np.array): The dataset to split.
        train_ratio (float): The ratio of the data to include in the training set.

    Returns:
        tuple: Two datasets - train and test.
    """
    # Shuffle the data
    random.shuffle(data)

    # Calculate the split index
    split_index = int(len(data) * train_ratio)

    # Split the data
    train_data = data[:split_index]
    test_data = data[split_index:]

    return train_data, test_data

# Split `celeb` into 70% train and 30% test
celeb_train_hog, celeb_test_hog = split_data(celeb, train_ratio=0.7)

# Split `ffhq` into 70% train and 30% test
ffhq_train_hog, ffhq_test_hog = split_data(ffhq, train_ratio=0.7)

# Split `attgan` into 70% train and 30% test
attgan_train_hog, attgan_test_hog = split_data(attgan, train_ratio=0.7)

# Split `stargan` into 70% train and 30% test
stargan_train_hog, stargan_test_hog = split_data(stargan, train_ratio=0.7)

# Split `gdwct` into 70% train and 30% test
gdwct_train_hog, gdwct_test_hog = split_data(gdwct, train_ratio=0.7)

# Split `stylegan2` into 70% train and 30% test_hog
stylegan2_train_hog, stylegan2_test_hog = split_data(stylegan2, train_ratio=0.7)

# Split `stylegan` into 70% train and 30% test_hog
stylegan_train_hog, stylegan_test_hog = split_data(stylegan, train_ratio=0.7)

# Convert to NumPy arrays if needed
celeb_train_hog, celeb_test_hog = np.array(celeb_train_hog), np.array(celeb_test_hog)
ffhq_train_hog, ffhq_test_hog = np.array(ffhq_train_hog), np.array(ffhq_test_hog)
attgan_train_hog, attgan_test_hog = np.array(attgan_train_hog), np.array(attgan_test_hog)
stargan_train_hog, stargan_test_hog = np.array(stargan_train_hog), np.array(stargan_test_hog)
gdwct_train_hog, gdwct_test_hog = np.array(gdwct_train_hog), np.array(gdwct_test_hog)
stylegan2_train_hog, stylegan2_test_hog = np.array(stylegan2_train_hog), np.array(stylegan2_test_hog)
stylegan_train_hog, stylegan_test_hog = np.array(stylegan_train_hog), np.array(stylegan_test_hog)

# Print results for verification
print(f"celeb_train: {len(celeb_train_hog)} images, celeb_test: {len(celeb_test_hog)} images")
print(f"ffhq_train: {len(ffhq_train_hog)} images, ffhq_test: {len(ffhq_test_hog)} images")
print(f"attgan_train: {len(attgan_train_hog)} images, attgan_test: {len(attgan_test_hog)} images")
print(f"stargan_train: {len(stargan_train_hog)} images, stargan_test: {len(stargan_test_hog)} images")
print(f"gdwct_train: {len(gdwct_train_hog)} images, gdwct_test: {len(gdwct_test_hog)} images")
print(f"stylegan2_train: {len(stylegan2_train_hog)} images, stylegan2_test: {len(stylegan2_test_hog)} images")
print(f"stylegan_train: {len(stylegan_train_hog)} images, stylegan_test: {len(stylegan_test_hog)} images")

########################################################################################################################################
#######################################divide into 60,10 train and val
#########################################################################################################################################
def extract_validation(train_data):
    """
    Extract every 10th sample from the training data and store it in a validation set.

    Parameters:
        train_data (list or np.array): The training dataset.

    Returns:
        tuple: Updated training dataset and validation dataset.
    """
    # Select every 10th sample for the validation set
    validation_data = train_data[::10]

    # Remove the selected samples from the training dataset
    updated_train_data = [train_data[i] for i in range(len(train_data)) if i % 10 != 0]

    return np.array(updated_train_data), np.array(validation_data)


# Perform the operation for each dataset
celeb_train_hog, celeb_val_hog = extract_validation(celeb_train_hog)
ffhq_train_hog, ffhq_val_hog = extract_validation(ffhq_train_hog)
attgan_train_hog, attgan_val_hog = extract_validation(attgan_train_hog)
stargan_train_hog, stargan_val_hog = extract_validation(stargan_train_hog)
gdwct_train_hog, gdwct_val_hog = extract_validation(gdwct_train_hog)
stylegan2_train_hog, stylegan2_val_hog = extract_validation(stylegan2_train_hog)
stylegan_train_hog, stylegan_val_hog = extract_validation(stylegan_train_hog)

# Print results for verification
print(f"celeb_train: {len(celeb_train_hog)} images, celeb_val: {len(celeb_val_hog)} images")
print(f"ffhq_train: {len(ffhq_train_hog)} images, ffhq_val: {len(ffhq_val_hog)} images")
print(f"attgan_train: {len(attgan_train_hog)} images, attgan_val: {len(attgan_val_hog)} images")
print(f"stargan_train: {len(stargan_train_hog)} images, stargan_val: {len(stargan_val_hog)} images")
print(f"gdwct_train: {len(gdwct_train_hog)} images, gdwct_val: {len(gdwct_val_hog)} images")
print(f"stylegan2_train: {len(stylegan2_train_hog)} images, stylegan2_val: {len(stylegan2_val_hog)} images")
print(f"stylegan_train: {len(stylegan_train_hog)} images, stylegan_val: {len(stylegan_val_hog)} images")
############################################################################################################################################################
#################################################concatenate the labels 0,1 real and fake
#############################################################################################################################################################


celeb_train_labels = np.zeros(len(celeb_train_hog), dtype=int)
ffhq_train_labels = np.zeros(len(ffhq_train_hog), dtype=int)
atta_train_labels = np.ones(len(attgan_train_hog), dtype=int)
star_train_labels = np.ones(len(stargan_train_hog), dtype=int)
gdwct_train_labels = np.ones(len(gdwct_train_hog), dtype=int)
stylegan2_train_labels = np.ones(len(stylegan2_train_hog), dtype=int)
stylegan_train_labels = np.ones(len(stylegan_train_hog), dtype=int)

# Concatenate all training datasets into a single `train` variable
train_hog = np.concatenate([celeb_train_hog, ffhq_train_hog, attgan_train_hog, stargan_train_hog, gdwct_train_hog, stylegan2_train_hog, stylegan_train_hog], axis=0)
train_labels=np.concatenate([celeb_train_labels, ffhq_train_labels, atta_train_labels, star_train_labels, gdwct_train_labels, stylegan2_train_labels,
                              stylegan_train_labels], axis=0)




celeb_test_labels = np.zeros(len(celeb_test_hog), dtype=int)
ffhq_test_labels = np.zeros(len(ffhq_test_hog), dtype=int)
atta_test_labels = np.ones(len(attgan_test_hog), dtype=int)
star_test_labels = np.ones(len(stargan_test_hog), dtype=int)
gdwct_test_labels = np.ones(len(gdwct_test_hog), dtype=int)
stylegan2_test_labels = np.ones(len(stylegan2_test_hog), dtype=int)
stylegan_test_labels = np.ones(len(stylegan_test_hog), dtype=int)

# Concatenate all testing datasets into a single `test` variable
test_hog = np.concatenate([celeb_test_hog, ffhq_test_hog, attgan_test_hog, stargan_test_hog, gdwct_test_hog, stylegan2_test_hog, stylegan_test_hog], axis=0)
test_labels = np.concatenate([celeb_test_labels, ffhq_test_labels, atta_test_labels, star_test_labels, gdwct_test_labels, stylegan2_test_labels,
                        stylegan_test_labels], axis=0)




celeb_val_labels = np.zeros(len(celeb_val_hog), dtype=int)
ffhq_val_labels = np.zeros(len(ffhq_val_hog), dtype=int)
atta_val_labels = np.ones(len(attgan_val_hog), dtype=int)
star_val_labels = np.ones(len(stargan_val_hog), dtype=int)
gdwct_val_labels = np.ones(len(gdwct_val_hog), dtype=int)
stylegan2_val_labels = np.ones(len(stylegan2_val_hog), dtype=int)
stylegan_val_labels = np.ones(len(stylegan_val_hog), dtype=int)

# Concatenate all validation datasets into a single `val` variable
val_hog = np.concatenate([celeb_val_hog, ffhq_val_hog, attgan_val_hog, stargan_val_hog, gdwct_val_hog, stylegan2_val_hog, stylegan_val_hog], axis=0)
val_labels = np.concatenate([celeb_val_labels, ffhq_val_labels, atta_val_labels, star_val_labels, gdwct_val_labels, stylegan2_val_labels,
                       stylegan_val_labels], axis=0)

# Print the results for verification
print(f"Total train: {len(train_hog)} images")
print(f"Total test: {len(test_hog)} images")
print(f"Total val: {len(val_hog)} images")


# Print results for verification
print(f"Train Labels: {len(train_labels)} ")
print(f"Test Labels: {len(test_labels)} ")
print(f"Val Labels: {len(val_labels)} ")



celeb shape: (5000, 224, 224, 3), dtype: uint8
ffhq shape: (5000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
celeb shape: (2500, 160, 160, 3), dtype: uint8
ffhq shape: (2500, 160, 160, 3), dtype: uint8
gdwct shape: (1000, 160, 160, 3), dtype: uint8
attagan shape: (1000, 160, 160, 3), dtype: uint8
stargan shape: (1000, 160, 160, 3), dtype: uint8
stylegan2 shape: (1000, 160, 160, 3), dtype: uint8
stylegan shape: (1000, 160, 160, 3), dtype: uint8
celeb_train: 1750 images, celeb_test: 750 images
ffhq_train: 1750 images, ffhq_test: 750 images
attgan_train: 700 images, attgan_test: 300 images
stargan_train: 700 images, stargan_test: 300 images
gdwct_train: 700 images, gdwct_test: 300 images
stylegan2_train: 700 images, stylegan2_test: 300 images
stylegan_train: 700 images, stylegan

In [8]:
# ============================================================
# DATASETS AND DATALOADERS
# ============================================================

train_dataset = DeepfakeCrossViTDataset(
    train_hog,
    train_labels,
    crossvit_transform
)

val_dataset = DeepfakeCrossViTDataset(
    val_hog,
    val_labels,
    crossvit_transform
)

test_dataset = DeepfakeCrossViTDataset(
    test_hog,
    test_labels,
    crossvit_transform
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training samples: 6300
Validation samples: 700
Testing samples: 3000
Training batches: 394
Validation batches: 44
Testing batches: 188


In [9]:
history = train_crossvit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=EPOCHS
)

Epoch 01/10 | Train loss: 0.2412 | Train accuracy: 0.8935 | Validation loss: 0.1018 | Validation accuracy: 0.9657
Epoch 02/10 | Train loss: 0.0445 | Train accuracy: 0.9844 | Validation loss: 0.0581 | Validation accuracy: 0.9814
Epoch 03/10 | Train loss: 0.0208 | Train accuracy: 0.9930 | Validation loss: 0.0645 | Validation accuracy: 0.9743
Epoch 04/10 | Train loss: 0.0231 | Train accuracy: 0.9914 | Validation loss: 0.0822 | Validation accuracy: 0.9786
Epoch 05/10 | Train loss: 0.0226 | Train accuracy: 0.9925 | Validation loss: 0.0578 | Validation accuracy: 0.9814
Epoch 06/10 | Train loss: 0.0091 | Train accuracy: 0.9970 | Validation loss: 0.0665 | Validation accuracy: 0.9800
Epoch 07/10 | Train loss: 0.0007 | Train accuracy: 1.0000 | Validation loss: 0.0793 | Validation accuracy: 0.9843
Epoch 08/10 | Train loss: 0.0002 | Train accuracy: 1.0000 | Validation loss: 0.0846 | Validation accuracy: 0.9829
Epoch 09/10 | Train loss: 0.0001 | Train accuracy: 1.0000 | Validation loss: 0.0888 | Va

In [10]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [11]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [ ]:
test_results, confusion, classification_report_text, predictions_df = (
    evaluate_crossvit(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)
for metric, value in test_results.items():
    if isinstance(value, (float, np.floating)):
        print(f"{metric:30s}: {value:.6f}")
    else:
        print(f"{metric:30s}: {value}")
print("\nConfusion Matrix:")
print(confusion)

print("\nFormat:")
print("[[TN, FP],")
print(" [FN, TP]]")
print("\nClassification Report:")
print(classification_report_text)
display(predictions_df.head())
test_results_df = pd.DataFrame([test_results])
display(test_results_df)


test_loss                     : 0.739426
accuracy                      : 0.988000
balanced_accuracy             : 0.988000
precision                     : 0.989960
recall_sensitivity            : 0.986000
specificity                   : 0.990000
f1_score                      : 0.987976
mcc                           : 0.976000
roc_auc                       : 0.998357
pr_auc                        : 0.998489
average_precision             : 0.998489
eer                           : 0.010000
eer_threshold                 : 0.855206
false_positive_rate           : 0.010000
false_negative_rate           : 0.014000
true_negatives                : 1485
false_positives               : 15
false_negatives               : 21
true_positives                : 1479
number_of_test_images         : 3000

Confusion Matrix:
[[1485   15]
 [  21 1479]]

Format:
[[TN, FP],
 [FN, TP]]

Classification Report:
              precision    recall  f1-score   support

        real     0.9861    0.9900    0.9880    

In [13]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 11.4%
Time Usage: 7.7 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [14]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 8.0%
Time Usage: 9.9 s
GPU Memory Used: 424.3 MB
Power Consumption: 93W


save the model

In [15]:
SAVE_DIR = (
    r"D:\thesis\results"
    r"\crossvit_15_160_dfc"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

CHECKPOINT_PATH = os.path.join(
    SAVE_DIR,
    "crossvit_15_checkpoint_dfc.pt"
)

torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "history":
            history,

        "model_name":
            MODEL_NAME,

        "input_size":
            INPUT_SIZE,

        "epochs":
            EPOCHS,

        "batch_size":
            BATCH_SIZE,

        "learning_rate":
            LEARNING_RATE,

        "num_classes":
            2,

        "label_mapping": {
            0: "real",
            1: "fake"
        },

        "position_embeddings_resized":
            True,

        "original_input_size":
            240
    },
    CHECKPOINT_PATH
)

print("CrossViT checkpoint saved:")
print(CHECKPOINT_PATH)

CrossViT checkpoint saved:
D:\thesis\results\crossvit_15_160_dfc\crossvit_15_checkpoint_dfc.pt


#load the model

In [16]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_9676\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [17]:
# ============================================================
# PYTORCH CROSSVIT WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# ============================================================
# DEVICE AND MODEL
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(
    test_loader.dataset
)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ============================================================
# GPU/MODEL WARM-UP
# ============================================================

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():

    for batch_index, batch in enumerate(
        test_loader
    ):

        if batch_index >= WARMUP_BATCHES:
            break

        images = batch[
            "pixel_values"
        ].to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        # timm CrossViT accepts the image tensor directly
        logits = model(images)

        # Materialize the output
        _ = logits


# Wait until all asynchronous GPU operations finish
if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ============================================================
# FIVE REPEATED INFERENCE RUNS
# ============================================================

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):

    print(
        f"\nStarting CrossViT resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous GPU work remains queued
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():

        for batch in test_loader:

            images = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            # CrossViT returns the logits tensor directly
            logits = model(images)

            last_logits = logits

            processed_images += images.size(0)


    # Ensure inference is complete before stopping timing
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )


    # Verify that at least one batch was processed
    if last_logits is not None:

        last_output_value = float(
            last_logits[-1, 0]
            .detach()
            .cpu()
            .item()
        )

    else:
        raise RuntimeError(
            "No images were processed during inference."
        )


    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number
    run_summary[
        "processed_images"
    ] = processed_images

    run_summary[
        "last_output_value"
    ] = last_output_value

    run_results.append(
        run_summary
    )


    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )


    del last_logits
    del logits


results_df = pd.DataFrame(
    run_results
)

print(
    "\nIndividual CrossViT profiling runs:"
)

display(results_df)

Device: cuda
Test images: 3000
Test batches: 188

Performing warm-up using 3 batches...
Warm-up completed.

Starting CrossViT resource run 1/5
Run 1: 5.14 seconds | 1.7150 ms/image | 583.09 images/s

Starting CrossViT resource run 2/5
Run 2: 5.38 seconds | 1.7949 ms/image | 557.14 images/s

Starting CrossViT resource run 3/5
Run 3: 4.46 seconds | 1.4869 ms/image | 672.53 images/s

Starting CrossViT resource run 4/5
Run 4: 4.52 seconds | 1.5052 ms/image | 664.38 images/s

Starting CrossViT resource run 5/5
Run 5: 4.56 seconds | 1.5192 ms/image | 658.23 images/s

Individual CrossViT profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,5.144988,1.714996,583.091773,5.347768,12.093750,6879.851562,6880.109614,6884.160156,0.258052,4.308594,...,11.755102,24.0,62.510204,98,51.326776,81.648,0.076246,1,3000,-4.537227
1,5.384603,1.794868,557.144118,4.969056,11.200000,6880.117188,6880.513404,6884.921875,0.396216,4.804688,...,11.764706,24.0,60.313725,99,47.668196,101.309,0.073281,2,3000,-4.537227
2,4.460770,1.486923,672.529631,4.691642,12.600000,6880.132812,6880.454215,6884.929688,0.321403,4.796875,...,11.720930,24.0,73.302326,98,46.556209,83.255,0.059547,3,3000,-4.537227
3,4.515458,1.505153,664.384388,4.057386,7.990625,6880.164062,6880.453480,6882.933594,0.289418,2.769531,...,11.727273,24.0,73.795455,100,45.370795,81.618,0.058623,4,3000,-4.537227
4,4.557681,1.519227,658.229467,5.054403,10.209375,6880.171875,6880.403054,6883.531250,0.231179,3.359375,...,11.727273,24.0,70.295455,95,53.574227,119.464,0.069273,5,3000,-4.537227


In [18]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("Cross-ViT RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


Cross-ViT RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,4.812700,0.422709,4.287838,5.337562
1,latency_ms_per_image,1.604233,0.140903,1.429279,1.779187
2,throughput_images_per_s,627.075875,53.041416,561.216259,692.935492
3,average_cpu_percent,4.824051,0.488281,4.217770,5.430333
4,peak_cpu_percent,10.818750,1.824381,8.553482,13.084018
5,average_ram_mb,6880.386753,0.159778,6880.188363,6880.585144
6,peak_ram_mb,6884.095312,0.873406,6883.010836,6885.179789
7,average_incremental_ram_mb,0.299253,0.063877,0.219940,0.378567
8,peak_incremental_ram_mb,4.007812,0.908558,2.879689,5.135936
9,average_gpu_memory_mb,2614.996869,0.019504,2614.972652,2615.021087



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 1.604 ± 0.141 (95% CI: 1.429–1.779)
peak_ram_mb: 6884.095 ± 0.873 (95% CI: 6883.011–6885.180)
peak_gpu_memory_mb: 2627.258 ± 0.000 (95% CI: 2627.258–2627.258)
average_gpu_utilization_percent: 68.043 ± 6.249 (95% CI: 60.285–75.802)
average_gpu_power_w: 48.899 ± 3.435 (95% CI: 44.634–53.165)


#genralization

In [20]:
#wild deepfake on dfc
print("\nTest results of DFC on wild deepfake dataset (Cross-ViT):")
test_dataset = DeepfakeCrossViTDataset(test_images,test_labels,crossvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, classification_report_text, predictions_df = (
    evaluate_crossvit(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)
for metric, value in test_results.items():
    if isinstance(value, (float, np.floating)):
        print(f"{metric:30s}: {value:.6f}")
    else:
        print(f"{metric:30s}: {value}")




Test results of DFC on wild deepfake dataset (Cross-ViT):
test_loss                     : 1.821671
accuracy                      : 0.735333
balanced_accuracy             : 0.526889
precision                     : 0.760838
recall_sensitivity            : 0.943778
specificity                   : 0.110000
f1_score                      : 0.842492
mcc                           : 0.091469
roc_auc                       : 0.611918
pr_auc                        : 0.811528
average_precision             : 0.811627
eer                           : 0.417185
eer_threshold                 : 0.999788
false_positive_rate           : 0.890000
false_negative_rate           : 0.056222
true_negatives                : 495
false_positives               : 4005
false_negatives               : 759
true_positives                : 12741
number_of_test_images         : 18000


In [22]:
#celeb on dfc
print("\nTest results of DFC on Celeb-DF(V2) dataset (Cross-ViT):")
test_dataset = DeepfakeCrossViTDataset(test_celeb,test_labels,crossvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, classification_report_text, predictions_df = (
    evaluate_crossvit(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)
for metric, value in test_results.items():
    if isinstance(value, (float, np.floating)):
        print(f"{metric:30s}: {value:.6f}")
    else:
        print(f"{metric:30s}: {value}")




Test results of DFC on Celeb-DF(V2) dataset (Cross-ViT):
test_loss                     : 1.374297
accuracy                      : 0.736957
balanced_accuracy             : 0.486586
precision                     : 0.901302
recall_sensitivity            : 0.796289
specificity                   : 0.176883
f1_score                      : 0.845548
mcc                           : -0.019697
roc_auc                       : 0.467434
pr_auc                        : 0.893631
average_precision             : 0.893668
eer                           : 0.520089
eer_threshold                 : 0.998710
false_positive_rate           : 0.823117
false_negative_rate           : 0.203711
true_negatives                : 101
false_positives               : 470
false_negatives               : 1098
true_positives                : 4292
number_of_test_images         : 5961


In [27]:
#FF++ on hog
print("\nTest results of dfc on FF++ dataset (Cross-ViT):")
test_dataset = DeepfakeCrossViTDataset(test_ff,test_ff_labels,crossvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, classification_report_text, predictions_df = (
    evaluate_crossvit(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)
for metric, value in test_results.items():
    if isinstance(value, (float, np.floating)):
        print(f"{metric:30s}: {value:.6f}")
    else:
        print(f"{metric:30s}: {value}")


Test results of dfc on FF++ dataset (Cross-ViT):
test_loss                     : 3.604804
accuracy                      : 0.460202
balanced_accuracy             : 0.514510
precision                     : 0.425135
recall_sensitivity            : 0.839961
specificity                   : 0.189058
f1_score                      : 0.564537
mcc                           : 0.037486
roc_auc                       : 0.497152
pr_auc                        : 0.409721
average_precision             : 0.411074
eer                           : 0.501420
eer_threshold                 : 0.998793
false_positive_rate           : 0.810942
false_negative_rate           : 0.160039
true_negatives                : 273
false_positives               : 1171
false_negatives               : 165
true_positives                : 866
number_of_test_images         : 2475


# FF++

LOAD THE DATASET

In [6]:
import os, cv2, numpy as np

FINAL_ROOT = r'D:\thesis\ff_final'

def load_split(split, cls):
    base = os.path.join(FINAL_ROOT, split, cls)
    nested, ids = [], []
    for vid_id in sorted(os.listdir(base)):
        d = os.path.join(base, vid_id)
        frames = [cv2.imread(os.path.join(d, f)) for f in sorted(os.listdir(d))]
        if frames:
            nested.append(frames); ids.append(vid_id)
    return nested, ids

# Main splits
ff_real_train_f, ff_real_train_ids = load_split('train', 'real')
ff_fake_train_f, ff_fake_train_ids = load_split('train', 'fake')
ff_real_val_f,   ff_real_val_ids   = load_split('val',   'real')
ff_fake_val_f,   ff_fake_val_ids   = load_split('val',   'fake')
ff_real_test_f,  ff_real_test_ids  = load_split('test',  'real')
ff_fake_test_f,  ff_fake_test_ids  = load_split('test',  'fake')

print("Reloaded main splits. Example shape:", np.shape(ff_real_train_f[0][0]))  # (160,160,3)
print("Real train videos:", len(ff_real_train_f), "| Fake train videos:", len(ff_fake_train_f))
import numpy as np


def combine_split(real_videos, fake_videos):
    """
    Combine all frames from the real and fake video groups.

    Labels:
        0 = Real
        1 = Fake
    """

    real_frames = [
        frame
        for video in real_videos
        for frame in video
        if frame is not None
    ]

    fake_frames = [
        frame
        for video in fake_videos
        for frame in video
        if frame is not None
    ]

    if not real_frames:
        raise ValueError("No real frames found.")

    if not fake_frames:
        raise ValueError("No fake frames found.")

    real_frames = np.stack(real_frames).astype(np.uint8)
    fake_frames = np.stack(fake_frames).astype(np.uint8)

    images = np.concatenate(
        [real_frames, fake_frames],
        axis=0
    )

    real_labels = np.zeros(
        len(real_frames),
        dtype=np.uint8
    )

    fake_labels = np.ones(
        len(fake_frames),
        dtype=np.uint8
    )

    labels = np.concatenate(
        [real_labels, fake_labels],
        axis=0
    )

    return images, labels
# Training data
train_ff, train_ff_labels = combine_split(
    ff_real_train_f,
    ff_fake_train_f
)

# Validation data
val_ff, val_ff_labels = combine_split(
    ff_real_val_f,
    ff_fake_val_f
)

# Testing data
test_ff, test_ff_labels = combine_split(
    ff_real_test_f,
    ff_fake_test_f
)
print("\nTRAIN")
print("Images:", train_ff.shape)
print("Labels:", train_ff_labels.shape)
print("Real:", np.sum(train_ff_labels == 0))
print("Fake:", np.sum(train_ff_labels == 1))

print("\nVALIDATION")
print("Images:", val_ff.shape)
print("Labels:", val_ff_labels.shape)
print("Real:", np.sum(val_ff_labels == 0))
print("Fake:", np.sum(val_ff_labels == 1))

print("\nTEST")
print("Images:", test_ff.shape)
print("Labels:", test_ff_labels.shape)
print("Real:", np.sum(test_ff_labels == 0))
print("Fake:", np.sum(test_ff_labels == 1))

print("\nData types")
print("Train images:", train_ff.dtype)
print("Train labels:", train_ff_labels.dtype)

Reloaded main splits. Example shape: (160, 160, 3)
Real train videos: 517 | Fake train videos: 320

TRAIN
Images: (4595, 160, 160, 3)
Labels: (4595,)
Real: 2948
Fake: 1647

VALIDATION
Images: (948, 160, 160, 3)
Labels: (948,)
Real: 499
Fake: 449

TEST
Images: (2475, 160, 160, 3)
Labels: (2475,)
Real: 1444
Fake: 1031

Data types
Train images: uint8
Train labels: uint8


In [9]:
# ============================================================
# DATASETS AND DATALOADERS
# ============================================================

train_dataset = DeepfakeCrossViTDataset(
    train_ff,
    train_ff_labels,
    crossvit_transform
)

val_dataset = DeepfakeCrossViTDataset(
    val_ff,
    val_ff_labels,
    crossvit_transform
)

test_dataset = DeepfakeCrossViTDataset(
    test_ff,
    test_ff_labels,
    crossvit_transform
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training samples: 4595
Validation samples: 948
Testing samples: 2475
Training batches: 288
Validation batches: 60
Testing batches: 155


In [10]:
history = train_crossvit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=EPOCHS
)

Epoch 01/10 | Train loss: 0.3386 | Train accuracy: 0.8392 | Validation loss: 0.2762 | Validation accuracy: 0.8734
Epoch 02/10 | Train loss: 0.1601 | Train accuracy: 0.9356 | Validation loss: 0.2206 | Validation accuracy: 0.9082
Epoch 03/10 | Train loss: 0.1076 | Train accuracy: 0.9571 | Validation loss: 0.3230 | Validation accuracy: 0.8829
Epoch 04/10 | Train loss: 0.0880 | Train accuracy: 0.9652 | Validation loss: 0.2110 | Validation accuracy: 0.9124
Epoch 05/10 | Train loss: 0.0581 | Train accuracy: 0.9765 | Validation loss: 0.2557 | Validation accuracy: 0.9030
Epoch 06/10 | Train loss: 0.0628 | Train accuracy: 0.9756 | Validation loss: 0.2995 | Validation accuracy: 0.8840
Epoch 07/10 | Train loss: 0.0508 | Train accuracy: 0.9830 | Validation loss: 0.2847 | Validation accuracy: 0.9040
Epoch 08/10 | Train loss: 0.0391 | Train accuracy: 0.9865 | Validation loss: 0.3053 | Validation accuracy: 0.8966
Epoch 09/10 | Train loss: 0.0484 | Train accuracy: 0.9817 | Validation loss: 0.2727 | Va

In [11]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [12]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [ ]:
test_results, confusion, classification_report_text, predictions_df = (
    evaluate_crossvit(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)
for metric, value in test_results.items():
    if isinstance(value, (float, np.floating)):
        print(f"{metric:30s}: {value:.6f}")
    else:
        print(f"{metric:30s}: {value}")
print("\nConfusion Matrix:")
print(confusion)

print("\nFormat:")
print("[[TN, FP],")
print(" [FN, TP]]")
print("\nClassification Report:")
print(classification_report_text)
display(predictions_df.head())
test_results_df = pd.DataFrame([test_results])
display(test_results_df)

test_loss                     : 0.331534
accuracy                      : 0.877980
balanced_accuracy             : 0.888494
precision                     : 0.795620
recall_sensitivity            : 0.951503
specificity                   : 0.825485
f1_score                      : 0.866608
mcc                           : 0.766099
roc_auc                       : 0.968198
pr_auc                        : 0.960703
average_precision             : 0.960722
eer                           : 0.094133
eer_threshold                 : 0.825861
false_positive_rate           : 0.174515
false_negative_rate           : 0.048497
true_negatives                : 1192
false_positives               : 252
false_negatives               : 50
true_positives                : 981
number_of_test_images         : 2475

Confusion Matrix:
[[1192  252]
 [  50  981]]

Format:
[[TN, FP],
 [FN, TP]]

Classification Report:
              precision    recall  f1-score   support

        real     0.959742  0.825485  0.887565   

In [14]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 17.7%
Time Usage: 7.3 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [15]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 14.4%
Time Usage: 9.5 s
GPU Memory Used: 424.3 MB
Power Consumption: 93W


#save the model

In [16]:
SAVE_DIR = (
    r"D:\thesis\results"
    r"\crossvit_15_160_ff"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

CHECKPOINT_PATH = os.path.join(
    SAVE_DIR,
    "crossvit_15_checkpoint_ff.pt"
)

torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "history":
            history,

        "model_name":
            MODEL_NAME,

        "input_size":
            INPUT_SIZE,

        "epochs":
            EPOCHS,

        "batch_size":
            BATCH_SIZE,

        "learning_rate":
            LEARNING_RATE,

        "num_classes":
            2,

        "label_mapping": {
            0: "real",
            1: "fake"
        },

        "position_embeddings_resized":
            True,

        "original_input_size":
            240
    },
    CHECKPOINT_PATH
)

print("CrossViT checkpoint saved:")
print(CHECKPOINT_PATH)

CrossViT checkpoint saved:
D:\thesis\results\crossvit_15_160_ff\crossvit_15_checkpoint_ff.pt


#load the model

In [17]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_41748\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [18]:
# ============================================================
# PYTORCH CROSSVIT WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# ============================================================
# DEVICE AND MODEL
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(
    test_loader.dataset
)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ============================================================
# GPU/MODEL WARM-UP
# ============================================================

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():

    for batch_index, batch in enumerate(
        test_loader
    ):

        if batch_index >= WARMUP_BATCHES:
            break

        images = batch[
            "pixel_values"
        ].to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        # timm CrossViT accepts the image tensor directly
        logits = model(images)

        # Materialize the output
        _ = logits


# Wait until all asynchronous GPU operations finish
if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ============================================================
# FIVE REPEATED INFERENCE RUNS
# ============================================================

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):

    print(
        f"\nStarting CrossViT resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous GPU work remains queued
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():

        for batch in test_loader:

            images = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            # CrossViT returns the logits tensor directly
            logits = model(images)

            last_logits = logits

            processed_images += images.size(0)


    # Ensure inference is complete before stopping timing
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )


    # Verify that at least one batch was processed
    if last_logits is not None:

        last_output_value = float(
            last_logits[-1, 0]
            .detach()
            .cpu()
            .item()
        )

    else:
        raise RuntimeError(
            "No images were processed during inference."
        )


    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number
    run_summary[
        "processed_images"
    ] = processed_images

    run_summary[
        "last_output_value"
    ] = last_output_value

    run_results.append(
        run_summary
    )


    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )


    del last_logits
    del logits


results_df = pd.DataFrame(
    run_results
)

print(
    "\nIndividual CrossViT profiling runs:"
)

display(results_df)

Device: cuda
Test images: 2475
Test batches: 155

Performing warm-up using 3 batches...
Warm-up completed.

Starting CrossViT resource run 1/5
Run 1: 4.29 seconds | 1.7345 ms/image | 576.53 images/s

Starting CrossViT resource run 2/5
Run 2: 3.73 seconds | 1.5061 ms/image | 663.95 images/s

Starting CrossViT resource run 3/5
Run 3: 4.06 seconds | 1.6419 ms/image | 609.05 images/s

Starting CrossViT resource run 4/5
Run 4: 3.99 seconds | 1.6120 ms/image | 620.35 images/s

Starting CrossViT resource run 5/5
Run 5: 3.96 seconds | 1.6003 ms/image | 624.86 images/s

Individual CrossViT profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,4.292901,1.734505,576.533225,4.470610,11.200000,5803.898438,5804.097377,5804.105469,0.198940,0.207031,...,11.428571,12.0,60.452381,92,46.577667,92.527,0.057368,1,2475,-2.246411
1,3.727714,1.506147,663.945786,4.686806,9.406250,5804.199219,5804.741211,5809.085938,0.541992,4.886719,...,0.000000,0.0,67.722222,89,45.386083,79.524,0.048557,2,2475,-2.246411
2,4.063706,1.641901,609.050026,4.584786,8.878125,5804.308594,5804.490029,5806.226562,0.181435,1.917969,...,11.368421,12.0,63.473684,100,47.917289,75.824,0.056332,3,2475,-2.246411
3,3.989678,1.611991,620.350862,3.926683,8.512500,5804.351562,5804.631310,5808.691406,0.279748,4.339844,...,0.000000,0.0,66.051282,100,47.830410,92.357,0.055202,4,2475,-2.246411
4,3.960863,1.600349,624.863840,4.767105,9.765625,5804.367188,5804.686883,5809.062500,0.319696,4.695312,...,11.368421,12.0,62.736842,100,46.338658,79.786,0.053002,5,2475,-2.246411


In [19]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("Cross-Vit RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


Cross-Vit RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,4.006972,0.203337,3.754496,4.259449
1,latency_ms_per_image,1.618979,0.082156,1.516968,1.720989
2,throughput_images_per_s,618.948748,31.459295,579.886874,658.010621
3,average_cpu_percent,4.487198,0.332478,4.074372,4.900024
4,peak_cpu_percent,9.552500,1.038924,8.262506,10.842494
5,average_ram_mb,5804.529362,0.258962,5804.207819,5804.850905
6,peak_ram_mb,5807.434375,2.207884,5804.692925,5810.175825
7,average_incremental_ram_mb,0.304362,0.144511,0.124928,0.483796
8,peak_incremental_ram_mb,3.209375,2.060416,0.651031,5.767719
9,average_gpu_memory_mb,2614.890895,0.335847,2614.473886,2615.307904



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 1.619 ± 0.082 (95% CI: 1.517–1.721)
peak_ram_mb: 5807.434 ± 2.208 (95% CI: 5804.693–5810.176)
peak_gpu_memory_mb: 2615.258 ± 0.000 (95% CI: 2615.258–2615.258)
average_gpu_utilization_percent: 64.087 ± 2.850 (95% CI: 60.549–67.626)
average_gpu_power_w: 46.810 ± 1.069 (95% CI: 45.483–48.137)


#gernalization

In [22]:
#wild deepfake on ff
print("\nTest results of FF++ on wild deepfake dataset crossvIt:")
test_dataset = DeepfakeCrossViTDataset(test_images,test_labels,crossvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, classification_report_text, predictions_df = (
    evaluate_crossvit(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)
for metric, value in test_results.items():
    if isinstance(value, (float, np.floating)):
        print(f"{metric:30s}: {value:.6f}")
    else:
        print(f"{metric:30s}: {value}")



Test results of FF++ on wild deepfake dataset crossvIt:
test_loss                     : 1.965754
accuracy                      : 0.494111
balanced_accuracy             : 0.472444
precision                     : 0.730487
recall_sensitivity            : 0.515778
specificity                   : 0.429111
f1_score                      : 0.604637
mcc                           : -0.047811
roc_auc                       : 0.462402
pr_auc                        : 0.730087
average_precision             : 0.730151
eer                           : 0.522000
eer_threshold                 : 0.627344
false_positive_rate           : 0.570889
false_negative_rate           : 0.484222
true_negatives                : 1931
false_positives               : 2569
false_negatives               : 6537
true_positives                : 6963
number_of_test_images         : 18000


In [24]:
#celeb on ff
print("\nTest results of FF++ on Celeb-df(v2) dataset (Cross-ViT):")
test_dataset = DeepfakeCrossViTDataset(test_celeb,test_labels,crossvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, classification_report_text, predictions_df = (
    evaluate_crossvit(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)
for metric, value in test_results.items():
    if isinstance(value, (float, np.floating)):
        print(f"{metric:30s}: {value:.6f}")
    else:
        print(f"{metric:30s}: {value}")



Test results of FF++ on Celeb-df(v2) dataset (Cross-ViT):
test_loss                     : 3.478081
accuracy                      : 0.395739
balanced_accuracy             : 0.632198
precision                     : 0.977054
recall_sensitivity            : 0.339703
specificity                   : 0.924694
f1_score                      : 0.504130
mcc                           : 0.167603
roc_auc                       : 0.719737
pr_auc                        : 0.958663
average_precision             : 0.958679
eer                           : 0.332701
eer_threshold                 : 0.001062
false_positive_rate           : 0.075306
false_negative_rate           : 0.660297
true_negatives                : 528
false_positives               : 43
false_negatives               : 3559
true_positives                : 1831
number_of_test_images         : 5961


In [26]:
#DFC on ff
print("\nTest results of FF++ on DFC dataset (Cross-ViT):")
test_dataset = DeepfakeCrossViTDataset(test_hog,test_labels,crossvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, classification_report_text, predictions_df = (
    evaluate_crossvit(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)
for metric, value in test_results.items():
    if isinstance(value, (float, np.floating)):
        print(f"{metric:30s}: {value:.6f}")
    else:
        print(f"{metric:30s}: {value}")



Test results of FF++ on DFC dataset (Cross-ViT):
test_loss                     : 2.543063
accuracy                      : 0.446333
balanced_accuracy             : 0.446333
precision                     : 0.467316
recall_sensitivity            : 0.767333
specificity                   : 0.125333
f1_score                      : 0.580873
mcc                           : -0.139993
roc_auc                       : 0.367514
pr_auc                        : 0.411735
average_precision             : 0.412558
eer                           : 0.591667
eer_threshold                 : 0.981280
false_positive_rate           : 0.874667
false_negative_rate           : 0.232667
true_negatives                : 188
false_positives               : 1312
false_negatives               : 349
true_positives                : 1151
number_of_test_images         : 3000
